# Nemotron GRPO — RL fine-tune of 0.86 LoRA adapter

**Goal:** push LB from 0.86 → 0.87-0.89 via REINFORCE-with-group-baseline (GRPO-lite) on the categories with dense reward signal.

**Why this might work:** SFT has plateaued (0.86 with original data, 0.80 with hybrid). The 0.86 policy already produces correct answers ~26-100% of the time on most categories — that's enough reward density for RL to refine.

**Why this might not work:** RL on a 30B-A3B Mamba-Hybrid in a 12-hr Kaggle session is tight. Could regress if KL collapses or rewards hack.

## Algorithm (GRPO-lite / RLOO with KL)

For each prompt:
1. Generate K=4 rollouts from policy π (HF model.generate)
2. Compute reward r_i for each (extract_boxed → match GT)
3. Group-relative advantage: A_i = r_i - mean(r_1..K)
4. Log-probs under policy π and reference π_ref (frozen 0.86)
5. Loss = -mean(A_i × log π(rollout_i)) + KL_COEF × KL(π || π_ref)
6. Update LoRA params, backprop, optimizer step

## Critical choices

- **No vLLM** — uses HF model directly for rollouts AND log-prob compute. Slower (~5 min per training step) but no infra mismatch with the training model.
- **Reference = frozen 0.86 LoRA** — implemented via PEFT's `disable_adapter()` context (reverts to base model for ref log-probs). No extra memory cost.
- **LoRA-only updates** — base model frozen, only LoRA weights move. Safer than full fine-tune.
- **Categories**: train ONLY on equation_numeric, cipher, numeral first (reward density ≥ 26%). Skip bit_manipulation/cryptarithm (1-2% reward density = no learning signal).

## Required Kaggle inputs

- `metric/nemotron-3-nano-30b-a3b-bf16` (model)
- NVIDIA metric utility script (auto-attached when competition metric enabled)
- **Your 0.86 LoRA adapter** — set `ADAPTER_PATH` in cell 4
- Training data (any of your existing per-category JSONL files)

## Settings

- Internet: OFF (all installs from metric utility script)
- GPU: RTX 6000 Pro Blackwell

## Expected runtime per session

- Model load + adapter: ~5 min
- Each GRPO step: ~5 min (rollout K=4 + 2× log-prob passes + backward)
- 12-hr session ≈ 130 steps ≈ 520 prompts processed
- Save checkpoint every 20 steps for safety

In [ ]:
# ============================================================
# 1. SETUP — extract NVIDIA metric utility script bundle (offline)
# ============================================================
# Same proven pattern as the RFT notebook.
import subprocess, sys, os, glob

CANDIDATE_BUNDLES = [
    "/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script",   # has vLLM
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script",
    "/kaggle/usr/lib/nvidia-metric-utility-script",                    # stripped, no vLLM
]
bundle = next((b for b in CANDIDATE_BUNDLES if os.path.isdir(b)), None)
if bundle is None:
    raise FileNotFoundError(
        "No NVIDIA utility-script bundle found. Enable the competition metric."
    )
print(f"[ok] Using bundle: {bundle}")

# Uninstall Kaggle's older torch
subprocess.run(
    "uv pip uninstall torch torchvision torchaudio || "
    "pip uninstall -y torch torchvision torchaudio || true",
    shell=True, check=False
)

# Extract bundle
print("Extracting bundle to /tmp ...")
subprocess.run(f"tar -cf - -C {bundle} . | tar -xf - -C /tmp", shell=True, check=False)

# ptxas binaries
for ptxas in ["/tmp/triton/backends/nvidia/bin/ptxas",
              "/tmp/triton/backends/nvidia/bin/ptxas-blackwell"]:
    if os.path.exists(ptxas):
        subprocess.run(f"chmod +x {ptxas}", shell=True)

# Prune sys.path, put /tmp FIRST
PRUNE_HINTS = ("ryanholbrook/nvidia_utility_script", "/kaggle/working/packages")
sys.path = [p for p in sys.path if not any(h in p for h in PRUNE_HINTS)]
sys.path = ["/tmp"] + [p for p in sys.path if p != "/tmp"]

# Evict cached modules so /tmp versions win
for _m in list(sys.modules):
    top = _m.split(".")[0]
    if top in ("torch","torchvision","torchaudio","torchgen","functorch",
               "transformers","tokenizers","safetensors","huggingface_hub",
               "accelerate","peft","datasets","triton",
               "mamba_ssm","causal_conv1d","flash_attn"):
        del sys.modules[_m]

# Verify
import torch, transformers
print(f"\n[ok] torch        {torch.__version__}  {torch.__file__}")
print(f"[ok] transformers {transformers.__version__}  {transformers.__file__}")
try:
    import peft
    print(f"[ok] peft         {peft.__version__}")
except ImportError:
    print(f"[ERR] peft NOT FOUND — GRPO requires peft for disable_adapter()")
try:
    import causal_conv1d, mamba_ssm
    print(f"[ok] mamba_ssm    {mamba_ssm.__version__}  ← Mamba fast path ON")
except ImportError as e:
    print(f"[warn] Mamba fast path OFF — {e}")

assert torch.__file__.startswith("/tmp/"), "torch must come from /tmp"
print(f"\n[ok] cell 1 done — CUDA: {torch.cuda.is_available()}")

In [ ]:
# ============================================================
# 2. IMPORTS + TRITON PTXAS FIX (multi-layer nuclear option)
# ============================================================
# The previous fix (env vars + chmod /tmp copies) didn't work because
# triton's NvidiaTool.from_path() is hard-coded to try the utility-script
# path FIRST, and that file is un-executable. We need to intercept at
# multiple layers:
#   (a) chmod the original file in place (sometimes Kaggle mounts allow it)
#   (b) copy + chmod a /tmp version (done before)
#   (c) MONKEY-PATCH NvidiaTool.from_path to redirect ptxas lookups to /tmp
#   (d) Invalidate triton's cached knob values

import os, shutil, stat, subprocess

PTXAS_SEARCH_DIRS = [
    "/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script/triton/backends/nvidia/bin",
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin",
    "/kaggle/usr/lib/nvidia-metric-utility-script/triton/backends/nvidia/bin",
]
TMP_PTXAS_DIR = "/tmp/triton_bin"
os.makedirs(TMP_PTXAS_DIR, exist_ok=True)

# (a) Try to chmod the original files in place — works if the mount allows
print("Step (a): chmod originals in place")
for src_dir in PTXAS_SEARCH_DIRS:
    if not os.path.isdir(src_dir): continue
    for binary in ["ptxas", "ptxas-blackwell"]:
        src = os.path.join(src_dir, binary)
        if os.path.exists(src):
            try:
                # Add execute bits if missing
                cur = os.stat(src).st_mode
                os.chmod(src, cur | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
                # Verify
                if os.access(src, os.X_OK):
                    print(f"  [ok] chmod {src}  → executable")
                else:
                    print(f"  [warn] chmod {src} silently ignored (read-only mount)")
            except Exception as e:
                print(f"  [warn] chmod {src} failed: {e.__class__.__name__}")

# (b) Copy + chmod to /tmp (always works)
print("\nStep (b): copy + chmod /tmp versions")
found = {}
for src_dir in PTXAS_SEARCH_DIRS:
    if not os.path.isdir(src_dir): continue
    for binary in ["ptxas", "ptxas-blackwell"]:
        src = os.path.join(src_dir, binary)
        if os.path.exists(src) and binary not in found:
            dst = os.path.join(TMP_PTXAS_DIR, binary)
            try:
                shutil.copy(src, dst)
                os.chmod(dst, 0o755)
                if os.access(dst, os.X_OK):
                    found[binary] = dst
                    print(f"  [ok] {binary} → {dst}")
            except Exception as e:
                print(f"  [warn] {binary}: {e}")

chosen = found.get("ptxas-blackwell") or found.get("ptxas")
print(f"\nUsing executable ptxas: {chosen}")

# Set env vars (still useful as belt+suspenders)
for v in ("TRITON_PTXAS_PATH", "TRITON_PTXAS_BLACKWELL_PATH",
          "TRITON_PTXAS_BIN", "TRITON_PTXAS"):
    os.environ[v] = chosen

# (c) MONKEY-PATCH triton's NvidiaTool.from_path to redirect.
# This is the layer that actually runs subprocess on the path — if we
# intercept it, any path triton tries gets redirected to /tmp.
print("\nStep (c): monkey-patch NvidiaTool.from_path")
try:
    from triton import knobs as _triton_knobs
    _orig_from_path = _triton_knobs.NvidiaTool.from_path

    @staticmethod
    def _patched_from_path(path):
        # If triton tries to use any read-only utility-script ptxas, redirect
        if path and ("nvidia_metric_utility_script" in path
                     or "nvidia_utility_script" in path
                     or "nvidia-metric-utility-script" in path):
            base = os.path.basename(path)  # "ptxas" or "ptxas-blackwell"
            redirect = os.path.join(TMP_PTXAS_DIR, base)
            if os.path.exists(redirect) and os.access(redirect, os.X_OK):
                # Substitute our path
                return _orig_from_path(redirect)
        return _orig_from_path(path)

    _triton_knobs.NvidiaTool.from_path = _patched_from_path
    print("  [ok] NvidiaTool.from_path patched — utility-script paths now redirect to /tmp")
except Exception as e:
    print(f"  [warn] monkey-patch failed: {e}")

# (d) Invalidate cached knob values + lru_cache on version lookup
print("\nStep (d): invalidate caches")
try:
    from triton import knobs as _kn
    for attr in ("ptxas", "ptxas_blackwell"):
        if attr in _kn.nvidia.__dict__:
            del _kn.nvidia.__dict__[attr]
            print(f"  [ok] cleared knobs.nvidia.{attr}")
    from triton.backends.nvidia.compiler import get_ptxas_version
    get_ptxas_version.cache_clear()
    print("  [ok] get_ptxas_version lru_cache cleared")
except Exception as e:
    print(f"  [info] cache invalidation partial: {e}")

# Sanity test the chosen ptxas BEFORE we let triton near it
print("\nSanity test: run chosen ptxas --version")
try:
    out = subprocess.check_output([chosen, "--version"], stderr=subprocess.STDOUT, timeout=10)
    print(f"  [ok] {out.decode().strip().splitlines()[0]}")
except Exception as e:
    print(f"  [fail] {e}")
    raise RuntimeError(
        "ptxas at /tmp still not executable. Cannot proceed — Mamba fast "
        "path needs a working ptxas for triton kernel compilation."
    )

# CUDA + tokenizer env
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Imports
import json, time, re, hashlib, random
from contextlib import nullcontext
from collections import Counter, defaultdict
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print(f"\nPyTorch  : {torch.__version__}")
print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("\n[ok] cell 2 done — triton should now resolve ptxas to a working binary")

In [ ]:
# ============================================================
# 3. GRPO CONFIG — paths + hyperparameters
# ============================================================
MODEL_PATH    = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

# *** EDIT THIS *** — your 0.86 LoRA adapter
ADAPTER_PATH  = "/kaggle/input/models/manish756/nvidia-adapter/transformers/default/7"

# Training data — point at any of your per-category JSONL dirs
DATA_DIR_CANDIDATES = [
    "/kaggle/input/datasets/manish756/nemotron-dataset/all_categorical_splits",
    "/kaggle/input/datasets/asharamkanderiwal/nvidia-dataset/all_categorical_splits",
]

# ============================================================
# Which categories to train on
# ============================================================
# Only categories where the 0.86 model is right OFTEN ENOUGH that GRPO
# gets useful gradient signal (need rewards > 0 in some rollouts).
# Skip bit_manipulation/cryptarithm (1-2% reward density = useless RL signal).
TRAIN_CATEGORIES = [
    "train_cot_numeral.jsonl",              # 100% — easy refinement
    "train_cot_unit_conversion.jsonl",      # high accuracy
    "train_cot_gravity.jsonl",              # high accuracy
    "train_cot_cipher.jsonl",               # 30-50% — good signal
    "train_cot_equation_numeric_deduce.jsonl", # 26% — densest hard-category signal
]

# ============================================================
# Rollout settings
# ============================================================
K_ROLLOUTS    = 4         # samples per prompt
ROLLOUT_TEMP  = 0.7       # sampling temperature for rollouts
ROLLOUT_TOP_P = 0.95
MAX_NEW_TOK   = 3072      # rollout length cap

# ============================================================
# GRPO/PPO hyperparameters — CONSERVATIVE (avoid divergence)
# ============================================================
LR              = 1e-6    # 50× lower than SFT! RL is unstable
KL_COEF         = 0.05    # strong KL anchor to 0.86 baseline
ADV_NORM        = True    # normalize advantages within group
MAX_GRAD_NORM   = 0.5     # tighter than SFT's 1.0

NUM_TRAIN_STEPS  = 200    # ~120 in a single 12-hr session (≈5 min/step)
PROMPTS_PER_STEP = 1      # process 1 prompt per gradient step (K rollouts each)
                          # (memory-tight — increasing to 2 may OOM)

# ============================================================
# Checkpointing
# ============================================================
SAVE_EVERY_N_STEPS = 20
OUTPUT_DIR  = "/kaggle/working/grpo_adapter"
CKPT_DIR    = "/kaggle/working/grpo_checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

# Random seed
random.seed(1337)
torch.manual_seed(1337)

print("=" * 60)
print("  GRPO Configuration")
print("=" * 60)
print(f"  Adapter (start) : {ADAPTER_PATH}")
print(f"  Categories      : {len(TRAIN_CATEGORIES)} cats with usable reward density")
print(f"  K rollouts      : {K_ROLLOUTS}")
print(f"  Sampling        : temp={ROLLOUT_TEMP} top_p={ROLLOUT_TOP_P} max_tok={MAX_NEW_TOK}")
print(f"  LR              : {LR}  (50× lower than SFT)")
print(f"  KL coef         : {KL_COEF}  (anchor strength)")
print(f"  Max grad norm   : {MAX_GRAD_NORM}")
print(f"  Train steps     : {NUM_TRAIN_STEPS}  (≈{NUM_TRAIN_STEPS*5/60:.1f} hrs)")
print(f"  Save every      : {SAVE_EVERY_N_STEPS} steps")

In [ ]:
# ============================================================
# 4. LOAD MODEL + 0.86 LoRA ADAPTER (policy + reference)
# ============================================================
# Strategy: one model in memory.
#   - LoRA enabled  = policy π (trainable)
#   - LoRA disabled = reference π_ref (frozen base)
# This saves 60GB vs loading two separate models.

# Verify adapter
adapter_cfg = os.path.join(ADAPTER_PATH, "adapter_config.json")
adapter_w   = os.path.join(ADAPTER_PATH, "adapter_model.safetensors")
assert os.path.exists(adapter_cfg) and os.path.exists(adapter_w), (
    f"Adapter not found at {ADAPTER_PATH}\nEdit ADAPTER_PATH in cell 3."
)
with open(adapter_cfg) as f:
    cfg = json.load(f)
print(f"Adapter  r={cfg.get('r')}  alpha={cfg.get('lora_alpha')}  "
      f"targets={cfg.get('target_modules')}")

# Tokenizer (left-padding for batched generation later if needed)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model in BF16
print("\nLoading base model in BF16 (~3-5 min)...")
t0 = time.time()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype       = torch.bfloat16,
    device_map        = "cuda",
    trust_remote_code = True,
)
print(f"Base loaded in {time.time()-t0:.0f}s "
      f"({torch.cuda.memory_allocated()/1e9:.1f} GB VRAM)")

# Load 0.86 adapter as PEFT model — KEEP IT TRAINABLE
print("Attaching 0.86 LoRA adapter (TRAINABLE) ...")
model = PeftModel.from_pretrained(
    base_model, ADAPTER_PATH, is_trainable=True
)
model.train()   # we'll train this

# Freeze base, train only LoRA
n_train, n_total = 0, 0
for n, p in model.named_parameters():
    n_total += p.numel()
    if "lora_" in n:
        p.requires_grad = True
        n_train += p.numel()
    else:
        p.requires_grad = False
print(f"Trainable params: {n_train/1e6:.1f}M / {n_total/1e9:.2f}B total  "
      f"(LoRA only)")

# Verify LoRA isn't zero (silent-load bug we hit before)
for n, p in model.named_parameters():
    if "lora_B" in n and p.requires_grad:
        ma = p.abs().mean().item()
        print(f"Sanity {n.split('base_model.model.')[-1][:55]} mean_abs={ma:.6f}")
        if ma < 1e-6:
            raise RuntimeError("lora_B is ~0 — adapter load failed!")
        break

# Quick sanity test
test_in = tokenizer("Say hello in three words.", return_tensors="pt").to("cuda")
with torch.no_grad():
    out = model.generate(**test_in, max_new_tokens=20,
                         do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
print("\nSanity:", tokenizer.decode(out[0, test_in.input_ids.shape[1]:],
                                     skip_special_tokens=True))
print(f"\nVRAM after model load: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("[ok] Model + adapter ready")

In [ ]:
# ============================================================
# 5. LOAD PROMPTS + REWARD FUNCTION
# ============================================================
data_dir = None
for c in DATA_DIR_CANDIDATES:
    if c and os.path.isdir(c) and any(
        os.path.exists(os.path.join(c, f)) for f in TRAIN_CATEGORIES):
        data_dir = c; break
assert data_dir, f"No data dir found; searched {DATA_DIR_CANDIDATES}"
print(f"Data dir: {data_dir}")

def extract_boxed(text):
    """Return LAST non-empty \\boxed{...} content."""
    matches = re.findall(r"\\boxed\{([^}]*)\}", text or "")
    non_empty = [m.strip() for m in matches if m.strip()]
    return non_empty[-1] if non_empty else None

def normalize_answer(s):
    if s is None: return ""
    s = str(s).strip().lower().strip("\"' ")
    return s.replace(" ", "")

def reward_fn(generated_text: str, gt_answer: str) -> float:
    """Binary 0/1 reward: 1.0 if extracted answer matches GT, else 0.0."""
    pred = extract_boxed(generated_text)
    if pred is None: return 0.0
    return 1.0 if normalize_answer(pred) == normalize_answer(gt_answer) else 0.0

# Load prompts (just user_msg + GT answer; we don't need the noisy CoT)
all_prompts = []
for fname in TRAIN_CATEGORIES:
    fp = os.path.join(data_dir, fname)
    if not os.path.exists(fp):
        print(f"  [skip] {fname}"); continue
    cat = fname.replace("train_cot_", "").replace(".jsonl", "")
    n_loaded = 0
    with open(fp) as f:
        for line in f:
            if not line.strip(): continue
            try:
                r = json.loads(line)
                msgs = [m for m in r["messages"] if m.get("role") != "system"]
                if not msgs or msgs[-1].get("role") != "assistant": continue
                user_text  = msgs[0]["content"]
                asst_text  = msgs[-1]["content"]
                gt_answer  = extract_boxed(asst_text)
                if gt_answer is None: continue
                all_prompts.append({
                    "category":  cat,
                    "user":      user_text,
                    "gt_answer": gt_answer,
                })
                n_loaded += 1
            except Exception:
                continue
    print(f"  {n_loaded:>5} from {fname}")

# Dedupe by user prompt
seen = set(); unique = []
for p in all_prompts:
    h = hashlib.md5(p["user"].encode()).hexdigest()
    if h not in seen:
        seen.add(h); unique.append(p)
all_prompts = unique
random.shuffle(all_prompts)
print(f"\nTotal unique prompts: {len(all_prompts)}")

cat_counts = Counter(p["category"] for p in all_prompts)
print("\nPer-category counts:")
for c, n in cat_counts.most_common():
    print(f"  {c:30s} {n:>5}")

# Test reward function
test_text = "Some reasoning... and the answer is \\boxed{42}"
print(f"\nReward test: extract from {test_text!r} vs GT '42' → {reward_fn(test_text, '42')}")
print(f"Reward test: extract from {test_text!r} vs GT '99' → {reward_fn(test_text, '99')}")

In [ ]:
# ============================================================
# 6. CORE GRPO PRIMITIVES — rollout + log-prob + advantages
# ============================================================
# Each function takes care of one piece. We compose them in cell 7.

def build_prompt_ids(user_text: str) -> torch.Tensor:
    """Apply chat template and tokenize. Returns [1, seq_len] tensor on CUDA."""
    msgs = [{"role": "user", "content": user_text}]
    try:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True,
            enable_thinking=True)
    except TypeError:
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids
    return ids.to("cuda")


@torch.no_grad()
def rollout(prompt_ids: torch.Tensor, k: int) -> tuple[torch.Tensor, list[str]]:
    """Generate K rollouts from the policy (LoRA enabled).
    Returns: (rollout_ids [k, total_len], decoded_text [k])
    """
    model.eval()
    # Expand prompt to k copies for batched generation
    batched_prompt = prompt_ids.repeat(k, 1)
    out = model.generate(
        batched_prompt,
        do_sample           = True,
        temperature         = ROLLOUT_TEMP,
        top_p               = ROLLOUT_TOP_P,
        max_new_tokens      = MAX_NEW_TOK,
        num_return_sequences = 1,  # already batched
        pad_token_id        = tokenizer.eos_token_id,
    )
    # out: [k, prompt_len + new_len]
    decoded = []
    p_len = prompt_ids.shape[1]
    for i in range(k):
        new = out[i, p_len:]
        decoded.append(tokenizer.decode(new, skip_special_tokens=True))
    return out, decoded


def log_probs_of_completion(input_ids: torch.Tensor, prompt_len: int,
                              use_adapter: bool = True) -> torch.Tensor:
    """
    Compute sum of log p(completion_token) for the COMPLETION region only.
    
    Args:
        input_ids: [batch, total_len] full sequence (prompt + completion)
        prompt_len: length of the prompt portion (we mask these out)
        use_adapter: if True, use policy (LoRA); if False, use reference (base)
    
    Returns: tensor [batch] of sum-log-probs over completion tokens.
    """
    ctx = nullcontext() if use_adapter else model.disable_adapter()
    with ctx:
        outputs = model(input_ids=input_ids, use_cache=False)
        logits = outputs.logits   # [batch, seq, vocab]
    
    # Shift: predict token[i+1] from logits at position i
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = input_ids[..., 1:].contiguous()
    
    # Compute log-prob of each true next-token
    log_probs = F.log_softmax(shift_logits.float(), dim=-1)
    token_log_probs = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)
    # [batch, seq-1]
    
    # Mask: only count tokens in the COMPLETION region.
    # Position i in shift_labels corresponds to input_ids[i+1].
    # Completion starts at input_ids[prompt_len], i.e. shift_labels index prompt_len-1.
    mask = torch.zeros_like(token_log_probs)
    mask[:, prompt_len - 1:] = 1.0
    
    # Per-sequence sum log-prob, averaged over completion-token count
    summed     = (token_log_probs * mask).sum(dim=-1)
    n_tokens   = mask.sum(dim=-1).clamp(min=1)
    return summed / n_tokens   # mean log-prob per completion token


def compute_advantages(rewards: torch.Tensor) -> torch.Tensor:
    """Group-relative advantage: r_i - mean(r). Optionally z-normalized.
    Args:  rewards [k]
    Returns: advantages [k]
    """
    adv = rewards - rewards.mean()
    if ADV_NORM and rewards.std() > 1e-6:
        adv = adv / (rewards.std() + 1e-6)
    return adv


print("[ok] GRPO primitives defined: rollout(), log_probs_of_completion(), compute_advantages()")

In [ ]:
# ============================================================
# 7. GRPO TRAINING LOOP
# ============================================================
# For each step:
#   1) Pick a prompt from the dataset
#   2) Rollout K completions (sampling, no grad)
#   3) Compute rewards
#   4) Compute group-relative advantages
#   5) Forward pass with grad ON to get policy log-probs
#   6) Forward pass with adapter DISABLED to get reference log-probs
#   7) Loss = -mean(adv * log_pi) + KL_COEF * (log_pi - log_pi_ref)
#   8) Backward, clip, step
#   9) Periodically save adapter

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Optimizer — AdamW on LoRA params only (Blackwell-stable)
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LR,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.0,
)
print(f"[ok] Optimizer: AdamW on {sum(p.numel() for p in trainable_params)/1e6:.1f}M params")

# Running metrics for live progress
metrics = {
    "step": [], "loss": [], "reward_mean": [], "kl": [],
    "accept_rate": [],
}

# Save initial state so we can always revert if RL breaks things
def save_adapter(step: int):
    path = os.path.join(CKPT_DIR, f"step_{step:04d}")
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    print(f"  [ckpt] saved adapter to {path}")

print("\n" + "=" * 60)
print("Starting GRPO training")
print("=" * 60 + "\n")

t_start = time.time()
n_seen  = 0
prompt_iter = iter(all_prompts)

for step in range(1, NUM_TRAIN_STEPS + 1):
    # Get next prompt (cycle through dataset)
    try:
        p = next(prompt_iter)
    except StopIteration:
        random.shuffle(all_prompts)
        prompt_iter = iter(all_prompts)
        p = next(prompt_iter)
    n_seen += 1
    
    t_step = time.time()
    prompt_ids = build_prompt_ids(p["user"])
    p_len = prompt_ids.shape[1]
    
    # --- 1) Rollout K samples ---
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        rollout_ids, rollout_texts = rollout(prompt_ids, K_ROLLOUTS)
    
    # --- 2) Compute rewards ---
    rewards = torch.tensor(
        [reward_fn(t, p["gt_answer"]) for t in rollout_texts],
        device="cuda", dtype=torch.float32
    )
    
    # If ALL rollouts wrong OR all correct, advantage is zero everywhere
    # → no learning signal. Skip step to save compute.
    if rewards.std() < 1e-6:
        step_time = time.time() - t_step
        print(f"  step {step:3d} [{p['category'][:18]:<18}]  "
              f"R={rewards.mean():.2f}  SKIP (no signal)  {step_time/60:.1f}m")
        continue
    
    # --- 3) Group-relative advantages ---
    advantages = compute_advantages(rewards)
    
    # --- 4) Policy log-probs (WITH grad) ---
    model.train()
    optimizer.zero_grad()
    log_pi = log_probs_of_completion(
        rollout_ids, prompt_len=p_len, use_adapter=True
    )
    
    # --- 5) Reference log-probs (no grad, adapter disabled) ---
    with torch.no_grad():
        log_ref = log_probs_of_completion(
            rollout_ids, prompt_len=p_len, use_adapter=False
        )
    
    # --- 6) Loss = -E[A * log_pi] + KL_COEF * (log_pi - log_ref) ---
    pg_loss = -(advantages.detach() * log_pi).mean()
    kl      = (log_pi - log_ref.detach()).mean()
    loss    = pg_loss + KL_COEF * kl
    
    # --- 7) Backward + clip + step ---
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, MAX_GRAD_NORM)
    optimizer.step()
    
    # --- 8) Logging ---
    accept_rate = (rewards > 0).float().mean().item()
    step_time   = time.time() - t_step
    metrics["step"].append(step)
    metrics["loss"].append(loss.item())
    metrics["reward_mean"].append(rewards.mean().item())
    metrics["kl"].append(kl.item())
    metrics["accept_rate"].append(accept_rate)
    
    print(f"  step {step:3d} [{p['category'][:18]:<18}]  "
          f"R={rewards.mean():.2f} ({accept_rate*100:.0f}% kept)  "
          f"loss={loss.item():+.4f}  KL={kl.item():+.3f}  "
          f"|g|={grad_norm:.2f}  {step_time:.0f}s")
    
    # NaN guard
    if not torch.isfinite(loss):
        print(f"  [HALT] non-finite loss at step {step} — stopping")
        break
    
    # --- 9) Periodic checkpoint ---
    if step % SAVE_EVERY_N_STEPS == 0:
        save_adapter(step)
        recent = metrics["reward_mean"][-SAVE_EVERY_N_STEPS:]
        print(f"  [progress] last {SAVE_EVERY_N_STEPS} steps: "
              f"avg R = {sum(recent)/len(recent):.3f}  "
              f"elapsed = {(time.time()-t_start)/60:.0f}m\n")
    
    # Safety: stop if running low on session time (leave 20 min buffer)
    if (time.time() - t_start) > (11.5 * 3600):
        print(f"  [TIME] approaching 12-hr limit — saving and stopping")
        save_adapter(step)
        break

# Final save
print(f"\n{'='*60}")
print(f"Training complete: {len(metrics['step'])} steps in "
      f"{(time.time()-t_start)/60:.0f} min")
print(f"Final reward mean (last 20 steps): "
      f"{sum(metrics['reward_mean'][-20:])/max(len(metrics['reward_mean'][-20:]),1):.3f}")
print(f"{'='*60}")
save_adapter(step)
model.save_pretrained(OUTPUT_DIR)
print(f"\nFinal adapter saved to {OUTPUT_DIR}")

In [ ]:
# ============================================================
# 8. EVALUATION + ZIP FOR SUBMISSION
# ============================================================
# Quick sanity check: did training help? Run 20 held-out prompts through
# the FINAL adapter and the ORIGINAL adapter, compare accuracy.

print("Final reward trajectory (smoothed over groups of 10 steps):")
if len(metrics["reward_mean"]) >= 10:
    chunks = [metrics["reward_mean"][i:i+10]
              for i in range(0, len(metrics["reward_mean"]), 10)]
    for i, c in enumerate(chunks):
        avg = sum(c) / len(c)
        bar = "█" * int(avg * 40)
        print(f"  steps {i*10+1:3d}-{(i+1)*10:3d}: R={avg:.3f} {bar}")
else:
    print(f"  Only {len(metrics['reward_mean'])} steps — not enough for trajectory")

# KL trajectory — should grow slowly. Spikes = divergence.
print("\nKL trajectory (last 20 steps):")
for i, kl in enumerate(metrics["kl"][-20:]):
    bar = "▌" * max(int(abs(kl) * 20), 1)
    sign = "+" if kl > 0 else "-"
    print(f"  step {metrics['step'][-20+i]:3d}: KL={sign}{abs(kl):.4f} {bar}")

# Zip the final adapter for Kaggle dataset upload
import shutil
zip_path = "/kaggle/working/grpo_adapter.zip"
shutil.make_archive(
    "/kaggle/working/grpo_adapter",  # output base name (no ext)
    "zip",
    OUTPUT_DIR,                       # source dir
)
print(f"\n[ok] Adapter zip ready: {zip_path}")
print(f"      Size: {os.path.getsize(zip_path)/1024/1024:.1f} MB")
print(f"\nNEXT STEPS:")
print(f"  1. Download {zip_path}")
print(f"  2. Upload as Kaggle dataset (name: 'manish-nemotron-grpo')")
print(f"  3. Submit via the competition submission notebook")
print(f"  4. Compare score to 0.86 baseline")